In [1]:
# ----------------------------
# Imports
# ----------------------------
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import ast 

In [2]:
# ----------------------------
# Project Paths
# ----------------------------
notebook_path = Path.cwd()  # notebook is running from notebooks/
project_root = notebook_path.parent / "ecg-ai-statistical-evaluation"  # project root
modules_dir = project_root / "modules"
data_dir = project_root / "data"
splits_dir = project_root / "splits"

# Make sure modules folder is in path for imports
sys.path.append(str(modules_dir))

# Import your download function
from organization import download_ptbxl, create_train_val_test_split

# ----------------------------
# Step 1: Download PTB-XL if missing
# ----------------------------
if not data_dir.exists() or not any(data_dir.iterdir()):
    print("Data folder is missing or empty. Downloading PTB-XL...")
    download_ptbxl(data_dir)
else:
    print(f"Data already exists at {data_dir}, skipping download.")

Data already exists at c:\Users\delga\OneDrive\Documents\personal_projects\DeepAF Automated Atrial Fibrillation Detection\ecg-ai-statistical-evaluation\data, skipping download.


In [3]:
# ----------------------------
# Step 2: Prepare labels and multilabel array
# ----------------------------

# Load the metadata CSV which contains patient IDs and their associated classes
df = pd.read_csv(data_dir / "ptbxl_database.csv")  # Load metadata

# Load the class mapping CSV 
scp_file = "data/scp_statements.csv"  # Path to SCP statements file
scp_statements = pd.read_csv(scp_file)

# Define the scp codes 
# The 'scp_codes' column contains string representations of dictionaries. We need to convert them to actual dictionaries for analysis.
df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)
# Fix the first column name which should be our scp_code
scp_statements = scp_statements.rename(columns={scp_statements.columns[0]: "scp_code"})
print(scp_statements.columns) # Check the columns again to confirm the rename

# Keep only diagnostic classes/codes
scp_diag = scp_statements[scp_statements["diagnostic"] == 1]

# Create mapping from code to superclass (scp_code to diagnostic_class)
code_to_superclass = dict(
    zip(scp_diag["scp_code"], scp_diag["diagnostic_class"])
)
print("Example mapping:")
print(list(code_to_superclass.items())[:10])


Index(['scp_code', 'description', 'diagnostic', 'form', 'rhythm',
       'diagnostic_class', 'diagnostic_subclass', 'Statement Category',
       'SCP-ECG Statement Description', 'AHA code', 'aECG REFID', 'CDISC Code',
       'DICOM Code'],
      dtype='object')
Example mapping:
[('NDT', 'STTC'), ('NST_', 'STTC'), ('DIG', 'STTC'), ('LNGQT', 'STTC'), ('NORM', 'NORM'), ('IMI', 'MI'), ('ASMI', 'MI'), ('LVH', 'HYP'), ('LAFB', 'CD'), ('ISC_', 'STTC')]


In [8]:
from organization import extract_superclasses

# Apply function to the dataframe
df["superclasses"] = df["scp_codes"].apply(lambda x: extract_superclasses(x, code_to_superclass))
# Flatten all superclasses into one list and get unique values
unique_superclasses = set(cls for sublist in df["superclasses"] for cls in sublist)

print("Unique superclasses in dataset:")
print(unique_superclasses)
print(f"Total unique superclasses: {len(unique_superclasses)}")

Unique superclasses in dataset:
{'STTC', 'MI', 'HYP', 'NORM', 'CD'}
Total unique superclasses: 5


In [9]:
# Get unique patients
patient_labels = df.groupby('patient_id')['superclasses'].first()

# Build multilabel matrix
all_classes = sorted({cls for sublist in patient_labels for cls in sublist})
Y = np.array([[1 if cls in patient_labels.loc[pid] else 0 for cls in all_classes] 
              for pid in patient_labels.index])

# ----------------------------
# Step 3: Create Train / Val / Test split
# ----------------------------
df, train_patients, val_patients, test_patients = create_train_val_test_split(
    df,
    Y,
    patient_id_col="patient_id",
    random_state=42,
    save_dir=splits_dir,   # Save here
    save=True              # Set to False if you don’t want CSVs
)

print("Split completed!")
print(f"Train patients: {len(train_patients):,}")
print(f"Validation patients: {len(val_patients):,}")
print(f"Test patients: {len(test_patients):,}")

# ----------------------------
# Step 4: Quick check
# ----------------------------
print(df['split'].value_counts())

Splits saved successfully to c:\Users\delga\OneDrive\Documents\personal_projects\DeepAF Automated Atrial Fibrillation Detection\ecg-ai-statistical-evaluation\splits
Split completed!
Train patients: 13,208
Validation patients: 2,830
Test patients: 2,831
split
train    15275
test      3288
val       3236
Name: count, dtype: int64
